# Modelado — comparación de modelos (forecast t+1)

Comparación de **regresión lineal**, **Random Forest** y **XGBoost** para predecir
`prod_pet` del mes siguiente (`y_next`).

**Protocolo (ADR-028):**
- Se **entrena en `train`** y se **compara en `val`**; `test` queda intacto.
- Métricas: **RMSE** (m³, penaliza errores grandes) y **R²** (varianza explicada).
- Todo el preprocesamiento (one-hot, imputación, normalización) se **ajusta solo
  en train** y se aplica a val → sin leakage val→train. Las features ya son
  leak-safe en el tiempo (ver notebook 03).

La lógica reutilizable vive en `ml/modeling.py`; acá solo se orquesta.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ml import modeling

pd.set_option("display.width", 120)

## 1. Dataset y matriz de features

`build_feature_matrix` arma el dataset (notebook 03) y aplica el one-hot de las
categóricas **ajustado en train**. Las features = todas las columnas menos las
claves (`idpozo`, `periodo`, `periodo_objetivo`, `split`) y el target `y_next`.

In [2]:
ds_enc, feature_cols = modeling.build_feature_matrix()

print(f"Filas: {len(ds_enc):,}  |  Features: {len(feature_cols)}")
print(ds_enc["split"].value_counts().reindex(["train", "val", "test"]).to_string())

Filas: 319,560  |  Features: 380
split
train    223621
val       47883
test      48056


## 2. Split train/val + preprocesamiento (anti-leakage)

- **Imputación** de NaN con la **mediana de train** (Random Forest y regresión
  lineal no toman NaN). Los NaN vienen de las features de historia (primeros
  meses de cada pozo).
- **Normalización** (media 0, desvío 1) **solo para la regresión lineal**
  (sensible a la escala); los árboles no la necesitan.

Ambos estadísticos (mediana, media/desvío) se ajustan **solo en train**.

In [3]:
X_train, y_train, X_val, y_val = modeling.split_train_val(ds_enc, feature_cols)
print("train:", X_train.shape, " | val:", X_val.shape)

train: (223621, 380)  | val: (47883, 380)


## 3. Entrenamiento y comparación en val

`train_eval_models` entrena los 3 modelos en train (imputado; + normalizado para
el lineal) y los evalúa en val. Agregamos la **persistencia** (ŷ(t+1) = prod_pet(t))
como referencia: es el baseline a batir (ADR-029).

In [4]:
tabla, modelos = modeling.train_eval_models(X_train, y_train, X_val, y_val)

pers = modeling.persistence_val(ds_enc)
comparacion = pd.concat(
    [tabla, pd.DataFrame([{"modelo": "persistencia (baseline)", **pers}])],
    ignore_index=True,
).sort_values("val_rmse").reset_index(drop=True)
comparacion.round(3)

,modelo,val_rmse,val_r2
0,xgboost,237.268,0.894
1,reg_lineal,244.362,0.887
2,persistencia (baseline),250.589,0.881
3,random_forest,257.281,0.875


**Lectura.**

- **XGBoost** es el mejor (RMSE más bajo, R² más alto) y **supera a la
  persistencia**; la **regresión lineal** también la supera por poco. El
  **Random Forest**, con estos hiperparámetros, queda **por debajo** de la
  persistencia.
- Los **R² son todos altos (~0.88–0.89)**: la mayor parte de la varianza de
  `prod_pet(t+1)` ya la explica `prod_pet(t)` (autoregresión fuerte), así que el
  margen de mejora sobre la persistencia es **chico** y se juega en los errores
  grandes (RMSE).
- Que el Random Forest quede atrás es típico de esta configuración (muchas
  dummies ralas + `max_features="sqrt"` + hojas grandes → promedia de más);
  mejora bastante con tuning.

**Próximos pasos:**
1. **Tuning** de XGBoost sobre val (profundidad, learning rate, n_estimators).
2. Probar **`log1p`** sobre el target: la fuerte asimetría infla el RMSE y suele
   mejorar con la transformación.
3. Loguear las corridas en **MLflow** (consistente con los baselines, ADR-030).